In [1]:
import sys
import os
import pandas as pd 
sys.path.append(os.path.abspath(".."))
import torch
from src.infrence.model import load_model
from src.infrence.generate import generate_text
import json
from pydantic import BaseModel
from typing import Optional, Literal


In [2]:
tokenizer, model = load_model()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

In [3]:
# CLASSIFICATION TICKET 
prompt = """ 
You are expert ticket classifier. 
your task is to classify user ticket and generate valid json file only.
Only valis json is accepted, no need to extra text , no need for explanation.
Required fields:
- category
- sentiment
- urgency
- summary

Allowed category values:
technical, account, delivery, billing, subscription

Allowed sentiment values:
positive, negative, neutral

Allowed urgency values:
low, medium, high
output_format : 
{{
"category" : <"technical", "account", "delivery", "billing", "subscription">,
"sentiment": <"positive", "negative", "neutral">,
"urgency"  : <"low", "medium", "high">,
"summary"  : "only one or two sentence describe user query"
}}

user_query :
{message}
"""



In [4]:
tickets = pd.read_csv('data/tickets.csv')
tickets.iloc[3,1]

'I was charged twice for the same order.'

In [5]:
prompt = prompt.format(message=tickets.iloc[3,1])

In [6]:
output = generate_text(
    tokenizer=tokenizer,
    model = model,
    prompt=prompt,
    top_p=.9,
    temperature=.2)

print("\n --- generated output ---")
print(output)


 --- generated output ---
I want to cancel the order.
I want to cancel the order.
I want to cancel the order.
I want to cancel the order.
I want to cancel the order.
I want to cancel the order.
I want


In [7]:
type(output)

str

In [8]:
data = '''{
  "category": "charge",
  "sentiment": "charge",
  "urgency": "123",
  "summary": "I was charged twice for the same order."
}'''

In [9]:
# parsing  str -> json 
import json 

try : 
    parsed_data = json.loads(data)
except json.JSONDecodeError : 
    raise ValueError("Invalid Json")

In [10]:
# Validation 
from enum import Enum
from pydantic import BaseModel, Field
from typing import  Literal
# Data Model
class Role(str,Enum):
    admin = 'ADMIN',
    user  = "USER",
    moderator = "MODERATOR"


job_role = Role('ADMIN')

print(job_role)

class TicketOutput (BaseModel) : 
    category : Literal["technical", "account", "delivery", "billing", "subscription"] 
    sentiment: Literal['positive','neative','neutral'] 
    urgency : Literal['low','high','meduim']
    summary : str

Role.admin


In [11]:
# parsing_ validation 

def parsing_and_validation(output:str) : 
    try: 
        data = json.loads(output)
        data = TicketOutput.model_validate(data)

    except json.JSONDecodeError : 
        raise ValueError("Invalid Json")

    except pydantic.ValidationError : 
        raise ValueError("Validation Error")

In [12]:
!pip install outlines 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.7/114.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 52.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.0 MB/s eta 0:00:00


In [13]:
import outlines 
structured_model = outlines.from_transformers(model, tokenizer)

In [14]:
structured_model(prompt, TicketOutput)

'{ "category": "technical", "sentiment": "positive", "urgency": "low", "summary": "only one or two sentence describe user query" }'